In [0]:
"""
Data Quality - Prepare Input Task
Adds row_id to input data and stores in Unity Catalog
"""
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import re

# Get parameters from notebook widgets
dbutils.widgets.text("project_id", "")
dbutils.widgets.text("project_name", "")
dbutils.widgets.text("job_name", "")
dbutils.widgets.text("catalog", "dbr_uat01_gdm")
dbutils.widgets.text("schema", "db_gdm_data_quality_app")
dbutils.widgets.text("results_schema", "db_gdm_dq_results")
dbutils.widgets.text("source_type", "csv")
dbutils.widgets.text("source_path", "")

PROJECT_ID = dbutils.widgets.get("project_id")
PROJECT_NAME = dbutils.widgets.get("project_name")
JOB_NAME = dbutils.widgets.get("job_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
#RESULTS_SCHEMA = dbutils.widgets.get("results_schema")
RESULTS_SCHEMA = "db_gdm_dq_results"
SOURCE_TYPE = dbutils.widgets.get("source_type")
SOURCE_PATH = dbutils.widgets.get("source_path")

# Sanitize names for table naming (remove special characters)
def sanitize_name(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name).lower()

project_name_clean = sanitize_name(PROJECT_NAME) if PROJECT_NAME else f"project_{PROJECT_ID}"
job_name_clean = sanitize_name(JOB_NAME) if JOB_NAME else f"job_{PROJECT_ID}"

print(f"=== Data Quality - Prepare Input ===")
print(f"Project ID: {PROJECT_ID}")
print(f"Project Name: {PROJECT_NAME} (sanitized: {project_name_clean})")
print(f"Job Name: {JOB_NAME} (sanitized: {job_name_clean})")
print(f"Source Type: {SOURCE_TYPE}")
print(f"Source Path: {SOURCE_PATH}")
print(f"Results Schema: {RESULTS_SCHEMA}")

# Read input data
if SOURCE_TYPE == "csv":
    print(f"Reading CSV from: {SOURCE_PATH}")
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(SOURCE_PATH)
elif SOURCE_TYPE == "unity_catalog":
    print(f"Reading from Unity Catalog: {SOURCE_PATH}")
    df = spark.table(SOURCE_PATH)
else:
    raise ValueError(f"Unsupported source type: {SOURCE_TYPE}")

print(f"Input data rows: {df.count()}")
print(f"Input data columns: {len(df.columns)}")

# Clean column names to remove invalid characters for Delta/Unity Catalog
invalid_chars = r'[ ,;{}()\n\t=]'
def clean_column(col):
    return re.sub(r'[^a-zA-Z0-9_]', '_', col)

new_columns = [clean_column(col) for col in df.columns]
df = df.toDF(*new_columns)

# Add row_id using monotonically_increasing_id
df_with_id = df.withColumn("row_id", F.monotonically_increasing_id())

# Reorder columns to have row_id first
columns = ["row_id"] + [col for col in df.columns]
df_with_id = df_with_id.select(*columns)

# Add execution timestamp
df_with_id = df_with_id.withColumn("execution_timestamp", F.current_timestamp())

# Output table name - using results schema with meaningful naming
output_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_input"

# Count rows before writing to avoid extra Spark jobs after write
row_count = df_with_id.count()

print(f"Writing to: {output_table}")

# Write to Unity Catalog
df_with_id.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

print(f"✅ Successfully wrote {row_count} rows to {output_table}")

# Display sample
print("\nSample data (first 5 rows):")
df_with_id.show(5, truncate=False)